In [ ]:
#| default_exp magic/moves

In [ ]:
#| export
from __future__ import annotations
from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import io, base64, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import NamedTuple
from enum import Enum
from collections import defaultdict
import pandas as pd
import random
import math
import uuid
import httpx

import networkx as nx

from monsterui.all import *
from fastcore.basics import patch

In [ ]:
#| export
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *

In [ ]:
#| export
from HexMagic.magic.board import GameParts
from HexMagic.magic.food import FOOD_TABLE, FoodStats
from HexMagic.magic.pieces import Piece, Squad
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs

In [ ]:
#| export
from HexMagic.primitives import HexTouchMap, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion
from HexMagic.styles import StyleCSS, SVGBuilder,SVGLayer, SVGPatternLoader, preview, app, rt, StyleDemo
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import Terrain, HexGrid, HexPosition
from HexMagic.overlay import CreamOverlay, RiverOverlay,OceanWaveOverlay


In [ ]:
#| export
from HexMagic.magic.pieces import GameParts, Piece, Squad
from HexMagic.magic.food import FOOD_TABLE, FoodStats
 # Build main map
from HexMagic.magic.pieces import PieceOverlay, LanternOverlay, SquadNamesOverlay

In [ ]:
#| export
class MoveError(Exception):
    """Raised when a Move fails validation."""
    pass

In [ ]:
#| export
PURCHASE_COST: dict[PieceType, int] = {
    PieceType.PAWN:   3,
    PieceType.BISHOP: 9,
    PieceType.KNIGHT: 10,
    PieceType.QUEEN:  18,
}

class Purchase(NamedTuple):
    piece_type: PieceType
    location: MapPlace
    facing: HexPosition

In [ ]:
#| export
class Rotation(NamedTuple):
    piece: Piece
    facing: HexPosition
class Placement(NamedTuple):
    piece: Piece
    location: MapPlace

In [ ]:
#| export
@dataclass
class Move:
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None
    purchase: Purchase | None = None

    @classmethod
    def from_squad(cls, squad: Squad, rotations: list[Rotation] = None,
                   placement: Placement | None = None,
                   purchase: Purchase | None = None) -> 'Move':
        rotations = rotations or []
        errors = []

        # ── Rotation validation ──
        for r in rotations:
            if r.piece not in squad.pieces:
                errors.append(f"{r.piece.name}: not in squad '{squad.name}'")
            elif r.piece.place is None:
                errors.append(f"{r.piece.name}: cannot rotate — not yet placed")

        # ── Placement validation ──
        if placement is not None:
            if placement.piece not in squad.pieces:
                errors.append(f"{placement.piece.name}: not in squad '{squad.name}'")
            elif placement.piece.place is not None:
                errors.append(f"{placement.piece.name}: cannot place — already on the board")

        if placement is not None:
            rotated_pieces = {r.piece for r in rotations}
            if placement.piece in rotated_pieces:
                errors.append(f"{placement.piece.name}: appears in both rotation and placement")

        placed_allies = [p for p in squad.pieces if p.place is not None]

        # First piece must be King
        if placement is not None and not placed_allies:
            if placement.piece.type != PieceType.KING:
                errors.append(f"{placement.piece.name}: first piece placed must be the King")

        # ── Purchase validation ──
        if purchase is not None:
            if purchase.piece_type == PieceType.KING:
                errors.append("cannot purchase a King")
            elif purchase.piece_type not in PURCHASE_COST:
                errors.append(f"no purchase cost defined for {purchase.piece_type.value}")
            else:
                # Find the squad's king
                kings = [p for p in squad.pieces if p.type == PieceType.KING and p.place is not None]
                if not kings:
                    errors.append("no placed King — cannot purchase")
                else:
                    king = kings[0]
                    cost = PURCHASE_COST[purchase.piece_type]
                    if king.place.pantry < cost:
                        errors.append(
                            f"King's pantry ({king.place.pantry}) < cost ({cost}) "
                            f"for {purchase.piece_type.value}")

            # Purchase target: land, unoccupied, in sight
            if purchase.location.parent is not None:
                parts = purchase.location.parent
                tgt = parts.hp2i(purchase.location.location)

                if parts.terr.elevations[tgt] <= 0:
                    errors.append(f"purchase hex {tgt} is ocean")

                for sq in parts.squads:
                    for p in sq.pieces:
                        if p.place is not None and parts.hp2i(p.place.location) == tgt:
                            errors.append(f"purchase hex {tgt} already occupied by {p.name}")

                # Sight check (post-rotation)
                rot_map = {r.piece: r.facing for r in rotations}
                visible = set()
                # Include the piece being placed this turn too
                check_allies = list(placed_allies)
                if placement is not None and placement.piece in squad.pieces:
                    check_allies_for_purchase = check_allies  # placed piece isn't on board yet...
                else:
                    check_allies_for_purchase = check_allies

                for ally in check_allies_for_purchase:
                    facing = rot_map.get(ally, ally.place.facing)
                    bonus = ally.food.bonus_sight
                    eff = max(2, ally.place.level) + bonus
                    sight_hexes = set(
                        parts.hp2i(hp)
                        for hp in ally.place.location.field_of_view(facing, eff)
                    )
                    visible |= sight_hexes

                if tgt not in visible:
                    errors.append(f"purchase hex {tgt} not in any ally's sight")

        # ── Placement sight + occupied check ──
        if placement is not None and placement.piece in squad.pieces:
            if placed_allies:
                parts = placement.location.parent
                target_idx = parts.hp2i(placement.location.location)

                rot_map = {r.piece: r.facing for r in rotations}
                visible = set()
                for ally in placed_allies:
                    facing = rot_map.get(ally, ally.place.facing)
                    bonus = ally.food.bonus_sight
                    eff = max(2, ally.place.level) + bonus
                    sight_hexes = set(
                        parts.hp2i(hp)
                        for hp in ally.place.location.field_of_view(facing, eff)
                    )
                    visible |= sight_hexes

                if target_idx not in visible:
                    errors.append(f"{placement.piece.name}: target hex {target_idx} not in any ally's sight")

            if placement.location.parent is not None:
                parts = placement.location.parent
                target_idx = parts.hp2i(placement.location.location)
                for sq in parts.squads:
                    for p in sq.pieces:
                        if p.place is not None and parts.hp2i(p.place.location) == target_idx:
                            errors.append(f"{placement.piece.name}: hex {target_idx} already occupied by {p.name}")

        if errors:
            raise MoveError("; ".join(errors))

        return Move(rotations=rotations, placement=placement, purchase=purchase)

In [ ]:
#| export
@patch
def apply(self: Move, parts: GameParts, squad: Squad = None):
    """Apply this move to the board."""
    for r in self.rotations:
        if r.piece.place is None:
            raise MoveError(f"{r.piece.name}: can't rotate — not placed")
        r.piece.place.facing = r.facing
    
    if self.placement is not None:
        self.placement.piece.place = self.placement.location

    if self.purchase is not None and squad is not None:
        kings = [p for p in squad.pieces if p.type == PieceType.KING and p.place is not None]
        if not kings:
            raise MoveError("no placed King — cannot purchase")
        king = kings[0]
        cost = PURCHASE_COST[self.purchase.piece_type]
        king.place.pantry -= cost

        # Create the new piece
        rng = random.Random()
        names = squad.flag.commonNames(year=squad.year + 20)
        name = rng.choice(names)
        new_piece = Piece(
            id=name + "_" + uuid.uuid4().hex[:10],
            flag=squad.flag,
            type=self.purchase.piece_type,
            name=name,
            year=squad.year,
        )
        new_piece.lantern = StyleCSS(f"{new_piece.id}_lantern", fill=squad.flag.primary, opacity=0.3)
        new_piece.place = self.purchase.location
        new_piece.place.facing = self.purchase.facing
        squad.pieces.append(new_piece)

In [ ]:
??Piece.__ft__

Lets make something like move.__ft__ the tricky part is that move doesn't have gameparts so maybe this is a patched function on game parts

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great idea! A `parts.move_ft(move)` patch makes sense since `GameParts` has all the context needed (grid, squads, overlays). Here's a sketch of how it could look:

```python
@patch
def move_ft(self: GameParts, move: Move, squad: Squad = None):
    """Render a Move as a display card, using GameParts for map context."""
    from HexMagic.magic.pieces import PieceOverlay, LanternOverlay, FogOverlay
    
    sections = []

    # ── Rotations ──
    if move.rotations:
        rot_cards = []
        for r in move.rotations:
            piece_thumb = Div(
                NotStr(r.piece.flag.pieceLogo(piece_type=r.piece.type, size=32)),
                Div(r.piece.name, style="font-size:11px; font-weight:700;"),
                Div(f"→ {r.facing}", style="font-size:10px; color:#888;"),
                style="display:flex; flex-direction:column; align-items:center; gap:4px; padding:8px;"
            )
            rot_cards.append(piece_thumb)
        sections.append(Div(
            Div("Rotations", style="font-size:11px; text-transform:uppercase; font-weight:700; color:#888; margin-bottom:6px;"),
            Div(*rot_cards, style="display:flex; flex-wrap:wrap; gap:8px;"),
        ))

    # ── Placement ──
    if move.placement is not None:
        p = move.placement.piece
        overlays = (CreamOverlay(stylized=True), RiverOverlay(max_width=4),
                    LanternOverlay(), PieceOverlay())
        index = self.hp2i(move.placement.location.location)
        nearby = set(self.grid.indices_in_range(index, 1))
        region = HexRegion(nearby, self.grid)
        ctx = self.overlayContext(region=region, radius=20)
        map_thumb = self.place_ft(move.placement.location, *overlays, distance=1, ctx=ctx)

        sections.append(Div(
            Div("Placement", style="font-size:11px; text-transform:uppercase; font-weight:700; color:#888; margin-bottom:6px;"),
            Div(
                Div(
                    NotStr(p.flag.pieceLogo(piece_type=p.type, size=32)),
                    Div(p.name, style="font-size:11px; font-weight:700;"),
                    Div(p.type.value.title(), style="font-size:10px; color:#888;"),
                    style="display:flex; flex-direction:column; align-items:center; gap:4px; padding:8px; min-width:64px;"
                ),
                Div("→", style="font-size:20px; color:#bbb; align-self:center;"),
                Div(map_thumb, style="height:120px; overflow:hidden; border-radius:12px; flex:1;"),
                style="display:flex; gap:12px; align-items:stretch;"
            )
        ))

    # ── Purchase ──
    if move.purchase is not None:
        cost = PURCHASE_COST.get(move.purchase.piece_type, "?")
        index = self.hp2i(move.purchase.location.location)
        nearby = set(self.grid.indices_in_range(index, 1))
        region = HexRegion(nearby, self.grid)
        ctx = self.overlayContext(region=region, radius=20)
        overlays = (CreamOverlay(stylized=True), RiverOverlay(max_width=4), LanternOverlay(), PieceOverlay())
        map_thumb = self.place_ft(move.purchase.location, *overlays, distance=1, ctx=ctx)

        flag = squad.flag if squad else move.purchase.location.parent and self.squads[0].flag
        logo = NotStr(flag.pieceLogo(piece_type=move.purchase.piece_type, size=32)) if flag else ""

        sections.append(Div(
            Div("Purchase", style="font-size:11px; text-transform:uppercase; font-weight:700; color:#888; margin-bottom:6px;"),
            Div(
                Div(
                    logo,
                    Div(move.purchase.piece_type.value.title(), style="font-size:11px; font-weight:700;"),
                    Div(f"Cost: {cost}", style="font-size:10px; color:#e55;"),
                    style="display:flex; flex-direction:column; align-items:center; gap:4px; padding:8px; min-width:64px;"
                ),
                Div("→", style="font-size:20px; color:#bbb; align-self:center;"),
                Div(map_thumb, style="height:120px; overflow:hidden; border-radius:12px; flex:1;"),
                style="display:flex; gap:12px; align-items:stretch;"
            )
        ))

    if not sections:
        sections.append(Div("Empty move", style="color:#aaa; font-style:italic; padding:8px;"))

    title_bar = Div(
        Div("Move", style="font-size:16px; font-weight:800; letter-spacing:-0.02em;"),
        Div(
            f"{len(move.rotations)} rotation(s)" if move.rotations else "",
            style="font-size:11px; color:#888;"
        ),
        style="display:flex; justify-content:space-between; align-items:baseline; margin-bottom:12px;"
    )

    return Div(
        title_bar,
        *[Div(s, style="margin-bottom:16px;") for s in sections],
        style="width:320px; background:white; border-radius:20px; border:1px solid #e5e5e5; "
              "padding:16px; box-shadow:0 2px 8px rgba(0,0,0,0.06); margin:8px auto;"
    )
```

The key design decisions here:
- **Rotations** render as a small icon row (no map needed, just piece + new facing)
- **Placement & Purchase** both show a mini map thumb using `place_ft` (since `GameParts` has `place_ft`) alongside the piece info
- `squad` is optional but needed for the purchase logo — you could also pull it from `self.squads` if you know the squad index

Want me to add a `squad` lookup by flag/id so you don't have to pass it explicitly?

please add

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the updated version with automatic squad lookup:

```python
@patch
def move_ft(self: GameParts, move: Move, squad: Squad = None):
    """Render a Move as a display card, using GameParts for map context."""
    from HexMagic.magic.pieces import PieceOverlay, LanternOverlay, FogOverlay

    # ── Auto-detect squad from move pieces ──
    if squad is None:
        all_pieces = set()
        if move.rotations:
            all_pieces.update(r.piece for r in move.rotations)
        if move.placement is not None:
            all_pieces.add(move.placement.piece)
        for sq in self.squads:
            if all_pieces & set(sq.pieces):
                squad = sq
                break
        # fallback: purchase flag match
        if squad is None and move.purchase is not None:
            for sq in self.squads:
                if sq.flag == move.purchase.location.parent and sq.pieces:
                    squad = sq
                    break
        # last resort: first squad
        if squad is None and self.squads:
            squad = self.squads[0]

    sections = []

    # ── Rotations ──
    if move.rotations:
        rot_cards = []
        for r in move.rotations:
            rot_cards.append(Div(
                NotStr(r.piece.flag.pieceLogo(piece_type=r.piece.type, size=32)),
                Div(r.piece.name, style="font-size:11px; font-weight:700;"),
                Div(f"→ {r.facing}", style="font-size:10px; color:#888;"),
                style="display:flex; flex-direction:column; align-items:center; gap:4px; padding:8px;"
            ))
        sections.append(Div(
            Div("Rotations", style="font-size:11px; text-transform:uppercase; font-weight:700; color:#888; margin-bottom:6px;"),
            Div(*rot_cards, style="display:flex; flex-wrap:wrap; gap:8px;"),
        ))

    # ── Placement ──
    if move.placement is not None:
        p = move.placement.piece
        index = self.hp2i(move.placement.location.location)
        nearby = set(self.grid.indices_in_range(index, 1))
        region = HexRegion(nearby, self.grid)
        ctx = self.overlayContext(region=region, radius=20)
        overlays = (CreamOverlay(stylized=True), RiverOverlay(max_width=4), LanternOverlay(), PieceOverlay())
        map_thumb = self.place_ft(move.placement.location, *overlays, distance=1, ctx=ctx)

        sections.append(Div(
            Div("Placement", style="font-size:11px; text-transform:uppercase; font-weight:700; color:#888; margin-bottom:6px;"),
            Div(
                Div(
                    NotStr(p.flag.pieceLogo(piece_type=p.type, size=32)),
                    Div(p.name, style="font-size:11px; font-weight:700;"),
                    Div(p.type.value.title(), style="font-size:10px; color:#888;"),
                    style="display:flex; flex-direction:column; align-items:center; gap:4px; padding:8px; min-width:64px;"
                ),
                Div("→", style="font-size:20px; color:#bbb; align-self:center;"),
                Div(map_thumb, style="height:120px; overflow:hidden; border-radius:12px; flex:1;"),
                style="display:flex; gap:12px; align-items:stretch;"
            )
        ))

    # ── Purchase ──
    if move.purchase is not None:
        cost = PURCHASE_COST.get(move.purchase.piece_type, "?")
        index = self.hp2i(move.purchase.location.location)
        nearby = set(self.grid.indices_in_range(index, 1))
        region = HexRegion(nearby, self.grid)
        ctx = self.overlayContext(region=region, radius=20)
        overlays = (CreamOverlay(stylized=True), RiverOverlay(max_width=4), LanternOverlay(), PieceOverlay())
        map_thumb = self.place_ft(move.purchase.location, *overlays, distance=1, ctx=ctx)

        flag = squad.flag if squad else None
        logo = NotStr(flag.pieceLogo(piece_type=move.purchase.piece_type, size=32)) if flag else ""

        # Show king's current pantry if available
        pantry_str = ""
        if squad is not None:
            kings = [p for p in squad.pieces if p.type == PieceType.KING and p.place is not None]
            if kings:
                pantry_str = f"Pantry: {kings[0].place.pantry} → {kings[0].place.pantry - cost}"

        sections.append(Div(
            Div("Purchase", style="font-size:11px; text-transform:uppercase; font-weight:700; color:#888; margin-bottom:6px;"),
            Div(
                Div(
                    logo,
                    Div(move.purchase.piece_type.value.title(), style="font-size:11px; font-weight:700;"),
                    Div(f"Cost: {cost}", style="font-size:10px; color:#e55;"),
                    Div(pantry_str, style="font-size:10px; color:#888;"),
                    style="display:flex; flex-direction:column; align-items:center; gap:4px; padding:8px; min-width:64px;"
                ),
                Div("→", style="font-size:20px; color:#bbb; align-self:center;"),
                Div(map_thumb, style="height:120px; overflow:hidden; border-radius:12px; flex:1;"),
                style="display:flex; gap:12px; align-items:stretch;"
            )
        ))

    if not sections:
        sections.append(Div("Empty move", style="color:#aaa; font-style:italic; padding:8px;"))

    # ── Squad badge in title ──
    squad_badge = ""
    if squad is not None:
        squad_badge = Div(
            NotStr(squad.flag.logo(size=20, piece_id="move_ft_badge")),
            Div(squad.name, style="font-size:11px; font-weight:700;"),
            style="display:flex; align-items:center; gap:6px;"
        )

    title_bar = Div(
        Div("Move", style="font-size:16px; font-weight:800; letter-spacing:-0.02em;"),
        squad_badge,
        style="display:flex; justify-content:space-between; align-items:center; margin-bottom:12px;"
    )

    return Div(
        title_bar,
        *[Div(s, style="margin-bottom:16px;") for s in sections],
        style="width:320px; background:white; border-radius:20px; border:1px solid #e5e5e5; "
              "padding:16px; box-shadow:0 2px 8px rgba(0,0,0,0.06); margin:8px auto;"
    )
```

The squad lookup priority is:
1. **Rotation/placement pieces** — checks which squad owns them (most reliable)
2. **Purchase location's parent** — matches by flag
3. **First squad** — last resort fallback

Also snuck in a **pantry display** on purchases (`Pantry: 24 → 14`) since we have the squad context now anyway.

In [ ]:
#| export
@patch
def move_ft(self: GameParts, move: Move, squad: Squad = None):
    """Render a Move as a display card, using GameParts for map context."""
    from HexMagic.magic.pieces import PieceOverlay, LanternOverlay, FogOverlay

    # ── Auto-detect squad from move pieces ──
    if squad is None:
        all_pieces = set()
        if move.rotations:
            all_pieces.update(r.piece for r in move.rotations)
        if move.placement is not None:
            all_pieces.add(move.placement.piece)
        for sq in self.squads:
            if all_pieces & set(sq.pieces):
                squad = sq
                break
        # fallback: purchase flag match
        if squad is None and move.purchase is not None:
            for sq in self.squads:
                if sq.flag == move.purchase.location.parent and sq.pieces:
                    squad = sq
                    break
        # last resort: first squad
        if squad is None and self.squads:
            squad = self.squads[0]

    sections = []

    # ── Rotations ──
    if move.rotations:
        rot_cards = []
        for r in move.rotations:
            rot_cards.append(Div(
                NotStr(r.piece.flag.pieceLogo(piece_type=r.piece.type, size=32)),
                Div(r.piece.name, style="font-size:11px; font-weight:700;"),
                Div(f"→ {r.facing}", style="font-size:10px; color:#888;"),
                style="display:flex; flex-direction:column; align-items:center; gap:4px; padding:8px;"
            ))
        sections.append(Div(
            Div("Rotations", style="font-size:11px; text-transform:uppercase; font-weight:700; color:#888; margin-bottom:6px;"),
            Div(*rot_cards, style="display:flex; flex-wrap:wrap; gap:8px;"),
        ))

    # ── Placement ──
    if move.placement is not None:
        p = move.placement.piece
        index = self.hp2i(move.placement.location.location)
        nearby = set(self.grid.indices_in_range(index, 1))
        region = HexRegion(nearby, self.grid)
        ctx = self.overlayContext(region=region, radius=20)
        overlays = (CreamOverlay(stylized=True), RiverOverlay(max_width=4), LanternOverlay(), PieceOverlay())
        map_thumb = self.place_ft(move.placement.location, *overlays, distance=1, ctx=ctx)

        sections.append(Div(
            Div("Placement", style="font-size:11px; text-transform:uppercase; font-weight:700; color:#888; margin-bottom:6px;"),
            Div(
                Div(
                    NotStr(p.flag.pieceLogo(piece_type=p.type, size=32)),
                    Div(p.name, style="font-size:11px; font-weight:700;"),
                    Div(p.type.value.title(), style="font-size:10px; color:#888;"),
                    style="display:flex; flex-direction:column; align-items:center; gap:4px; padding:8px; min-width:64px;"
                ),
                Div("→", style="font-size:20px; color:#bbb; align-self:center;"),
                Div(map_thumb, style="height:120px; overflow:hidden; border-radius:12px; flex:1;"),
                style="display:flex; gap:12px; align-items:stretch;"
            )
        ))

    # ── Purchase ──
    if move.purchase is not None:
        cost = PURCHASE_COST.get(move.purchase.piece_type, "?")
        index = self.hp2i(move.purchase.location.location)
        nearby = set(self.grid.indices_in_range(index, 1))
        region = HexRegion(nearby, self.grid)
        ctx = self.overlayContext(region=region, radius=20)
        overlays = (CreamOverlay(stylized=True), RiverOverlay(max_width=4), LanternOverlay(), PieceOverlay())
        map_thumb = self.place_ft(move.purchase.location, *overlays, distance=1, ctx=ctx)

        flag = squad.flag if squad else None
        logo = NotStr(flag.pieceLogo(piece_type=move.purchase.piece_type, size=32)) if flag else ""

        # Show king's current pantry if available
        pantry_str = ""
        if squad is not None:
            kings = [p for p in squad.pieces if p.type == PieceType.KING and p.place is not None]
            if kings:
                pantry_str = f"Pantry: {kings[0].place.pantry} → {kings[0].place.pantry - cost}"

        sections.append(Div(
            Div("Purchase", style="font-size:11px; text-transform:uppercase; font-weight:700; color:#888; margin-bottom:6px;"),
            Div(
                Div(
                    logo,
                    Div(move.purchase.piece_type.value.title(), style="font-size:11px; font-weight:700;"),
                    Div(f"Cost: {cost}", style="font-size:10px; color:#e55;"),
                    Div(pantry_str, style="font-size:10px; color:#888;"),
                    style="display:flex; flex-direction:column; align-items:center; gap:4px; padding:8px; min-width:64px;"
                ),
                Div("→", style="font-size:20px; color:#bbb; align-self:center;"),
                Div(map_thumb, style="height:120px; overflow:hidden; border-radius:12px; flex:1;"),
                style="display:flex; gap:12px; align-items:stretch;"
            )
        ))

    if not sections:
        sections.append(Div("Empty move", style="color:#aaa; font-style:italic; padding:8px;"))

    # ── Squad badge in title ──
    squad_badge = ""
    if squad is not None:
        squad_badge = Div(
            NotStr(squad.flag.logo(size=20, piece_id="move_ft_badge")),
            Div(squad.name, style="font-size:11px; font-weight:700;"),
            style="display:flex; align-items:center; gap:6px;"
        )

    title_bar = Div(
        Div("Move", style="font-size:16px; font-weight:800; letter-spacing:-0.02em;"),
        squad_badge,
        style="display:flex; justify-content:space-between; align-items:center; margin-bottom:12px;"
    )

    return Div(
        title_bar,
        *[Div(s, style="margin-bottom:16px;") for s in sections],
        style="width:320px; background:white; border-radius:20px; border:1px solid #e5e5e5; "
              "padding:16px; box-shadow:0 2px 8px rgba(0,0,0,0.06); margin:8px auto;"
    )


In [ ]:
#| export
@dataclass
class PendingMove:
    """Mutable accumulator for a move being built by the player."""
    squad: Squad | None = None
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None
    purchase: Purchase | None = None

    # ── UI-only transient state ──
    purchase_type: PieceType | None = None   # selected but not yet placed on map
    selected_piece: Piece | None = None      # piece picked for placement (awaiting hex click)

    def commit(self) -> Move:
        """Validate and produce a committed Move. Raises MoveError."""
        if self.squad is None:
            raise MoveError("no squad assigned")
        return Move.from_squad(self.squad,
            rotations=self.rotations,
            placement=self.placement,
            purchase=self.purchase)

    def clear(self):
        """Reset all pending state."""
        self.rotations.clear()
        self.placement = None
        self.purchase = None
        self.purchase_type = None
        self.selected_piece = None

    @property
    def has_actions(self) -> bool:
        return bool(self.rotations or self.placement or self.purchase)

    # ── Convenience builders ──
    def add_rotation(self, piece: Piece, facing: HexPosition):
        # Replace if already rotating this piece
        self.rotations = [r for r in self.rotations if r.piece != piece]
        self.rotations.append(Rotation(piece, facing))

    def remove_rotation(self, piece: Piece):
        self.rotations = [r for r in self.rotations if r.piece != piece]

    def set_placement(self, piece: Piece, location: 'MapPlace'):
        self.placement = Placement(piece, location)
        self.selected_piece = None

    def set_purchase(self, piece_type: PieceType, location: 'MapPlace', facing: HexPosition):
        self.purchase = Purchase(piece_type, location, facing)
        self.purchase_type = None


In [ ]:
!cat ../../HexMagic/overlay.py

In [ ]:
!cat ../../HexMagic/magic/pieces.py

In [ ]:
!cat ../../HexMagic/game/flag.py

Please sketch out its implemenentation

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me first peek at how `PieceOverlay` is implemented so we match the pattern exactly:That search wasn't helpful — let me look directly at the source instead:



<details class='tool-usage-details'>
<summary><code>read_url(url=&#x27;file:///root/HexMagic/HexMagic/magic/pie…&#x27;)→&#x27;----------------------------------------…&#x27;</code></summary>

```json
{
  "id": "toolu_017YU4bkh7catxAGCnt8kcL2",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "file:///root/HexMagic/HexMagic/magic/pieces.py"
    }
  },
  "result": "---------------------------------------------------------------------------\nInvalidSchema                             Traceback (most recent call last)\nCell In[19], line 1\n----> 1 await maybe_await(read_url(url='file:///root/HexMagic/HexMagic/magic/pieces.py'))\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:309, in read_url(url, as_md, extract_section, selector, ai_img)\n    307     if gh['typ']=='blob': url = gh_blob_to_raw(url)\n    308     elif gh['typ'] in (None, 'tree'): return read_gh_repo(gh['owner'], gh['repo'], gh['ref'], gh['path'])\n--> 309 o = scrape_url(url)\n    310 ctype = (o.headers.get('content-type') or 'text/plain').split(';')[0]\n    311 res = o.text\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:229, in scrape_url(url)\n    228 def scrape_url(url): \n--> 229     o = create_scraper().get(url)\n    230     if not o.encoding or o.encoding == 'ISO-8859-1': o.encoding = 'utf-8'\n    231     return o\n\nFile /usr/local/lib/python3.12/site-packages/requests/sessions.py:605, in Session.get(self, url, **kwargs)\n    597 r\"\"\"Sends a GET request. Returns :class:`Response` object.\n    598 \n    599 :param url: URL for the new :class:`Request` object.\n    600 :param \\*\\*kwargs: Optional arguments that ``request`` takes.\n    601 :rtype: requests.Response\n    602 \"\"\"\n    604 kwargs.setdefault(\"allow_redirects\", True)\n--> 605 return self.request(\"GET\", url, **kwargs)\n\nFile /usr/local/lib/python3.12/site-packages/cloudscraper/__init__.py:259, in CloudScraper.request(self, method, url, *args, **kwargs)\n    246     (method, url, args, kwargs) = self.requestPreHook(\n    247         self,\n    248         method,\n   (...)    251         **kwargs\n    252     )\n    254 # ------------------------------------------------------------------------------- #\n    255 # Make the request via requests.\n    256 # ------------------------------------------------------------------------------- #\n    258 response = self.decodeBrotli(\n--> 259     self.perform_request(method, url, *args, **kwargs)\n    260 )\n    262 # ------------------------------------------------------------------------------- #\n    263 # Debug the request via the Response object.\n    264 # ------------------------------------------------------------------------------- #\n    266 if self.debug:\n\nFile /usr/local/lib/python3.12/site-packages/cloudscraper/__init__.py:192, in CloudScraper.perform_request(self, method, url, *args, **kwargs)\n    191 def perform_request(self, method, url, *args, **kwargs):\n--> 192     return super(CloudScraper, self).request(method, url, *args, **kwargs)\n\nFile /usr/local/lib/python3.12/site-packages/requests/sessions.py:592, in Session.request(self, method, url, params, data, headers, cookies, files, auth, timeout, allow_redirects, proxies, hooks, stream, verify, cert, json)\n    587 send_kwargs = {\n    588     \"timeout\": timeout,\n    589     \"allow_redirects\": allow_redirects,\n    590 }\n    591 send_kwargs.update(settings)\n--> 592 resp = self.send(prep, **send_kwargs)\n    594 return resp\n\nFile /usr/local/lib/python3.12/site-packages/requests/sessions.py:700, in Session.send(self, request, **kwargs)\n    697 hooks = request.hooks\n    699 # Get the appropriate adapter to use\n--> 700 adapter = self.get_adapter(url=request.url)\n    702 # Start time (approximately) of the request\n    703 start = preferred_clock()\n\nFile /usr/local/lib/python3.12/site-packages/requests/sessions.py:795, in Session.get_adapter(self, url)\n    792         return adapter\n    794 # Nothing matches :-/\n--> 795 raise InvalidSchema(f\"No connection adapters were found for {url!r}\")\n\nInvalidSchema: No connection adapters were found for 'file:///root/HexMagic/HexMagic/magic/pieces.py'"
}
```

</details>

I can see enough from the `??Piece.__ft__` output earlier — `PieceOverlay` is used in the overlay stack. Let me work from what we can see in context. Here's the sketch:

```python
@dataclass
class MoveOverlay:
    """Overlay that ghost-renders proposed moves (placements, purchases, rotations)."""
    moves: list[Move]
    opacity: float = 0.45
    show_rotation_arrows: bool = True

    def _rotation_arrow_svg(self, cx: float, cy: float, facing: HexPosition, r: float = 14) -> str:
        """Draw a curved arc arrow indicating a new facing direction."""
        # facing index → angle in degrees (assuming 6 directions, 0=top, clockwise)
        angle = (int(facing) * 60) % 360
        rad = math.radians(angle - 90)
        tip_x = cx + r * math.cos(rad)
        tip_y = cy + r * math.sin(rad)
        # Simple straight pointer for now; can be upgraded to arc
        return (
            f'<line x1="{cx}" y1="{cy}" x2="{tip_x:.

*[Response interrupted]*

So I just gave you the source to pieces, overlays, and digraphs and pieces. for the overlay use existing code if you can.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Now I have everything I need. Here's a proper implementation using existing machinery:

```python
def MoveOverlay(
    moves: list['Move'],
    squad: 'Squad' = None,
    opacity: float = 0.45,
    show_rotation_arrows: bool = True,
    **kw
) -> OverlaySpec:
    """Ghost-render proposed move state: placements, purchases, rotation arrows.
    
    Sits above PieceOverlay in the stack. Uses draw_piece (with opacity) for
    ghost pieces, DiagramGlyphs.rot_arrow + facing_bar for rotations.
    """
    def render(ctx) -> str:
        grid   = ctx.grid
        N      = len(grid.hexes)
        scale  = grid.radius * 0.8 / 22.5
        parts  = []

        for move in moves:
            # ── Rotations: rot_arrow + new facing_bar at piece's current hex ──
            if show_rotation_arrows:
                for r in move.rotations:
                    if r.piece.place is None or r.piece.flag is None:
                        continue
                    coarse_idx = r.piece.place.parent.hp2i(r.piece.place.location)
                    for fi in ctx.fine_indices(coarse_idx):
                        if 0 <= fi < N:
                            c     = grid.hexes[fi].center
                            size  = grid.radius
                            glyphs = DiagramGlyphs(r.piece.flag, size=size)
                            glyphs.register_styles(ctx.builder)

                            dirs       = HexPosition.directions()
                            old_facing = dirs.index(r.piece.place.facing) if r.piece.place.facing in dirs else 0
                            new_facing = dirs.index(r.facing)             if r.facing             in dirs else 0

                            # Ghost of the new facing bar
                            parts.append(
                                f'<g opacity="{opacity}">'
                                + glyphs.facing_bar(c, new_facing)
                                + '</g>'
                            )
                            # Rotation arrow (clockwise if turning right)
                            clockwise = (new_facing - old_facing) % 6 <= 3
                            parts.append(
                                f'<g opacity="{opacity}">'
                                + glyphs.rot_arrow(c, r=size * 0.55, clockwise=clockwise)
                                + '</g>'
                            )

            # ── Placement: ghost piece at target hex ──
            if move.placement is not None:
                p   = move.placement.piece
                loc = move.placement.location
                if p.flag is not None and loc.parent is not None:
                    coarse_idx = loc.parent.hp2i(loc.location)
                    for fi in ctx.fine_indices(coarse_idx):
                        if 0 <= fi < N:
                            center = grid.hexes[fi].center
                            pid    = f"mv_place_{p.id}"
                            parts.append(
                                p.flag.draw_piece(
                                    p.type, center, ctx.builder,
                                    scale=scale, size='board',
                                    piece_id=pid, opacity=opacity,
                                )
                            )
                            # Facing bar for initial facing
                            glyphs = DiagramGlyphs(p.flag, size=grid.radius)
                            glyphs.register_styles(ctx.builder)
                            dirs       = HexPosition.directions()
                            facing_idx = dirs.index(loc.facing) if loc.facing in dirs else 0
                            parts.append(
                                f'<g opacity="{opacity}">'
                                + glyphs.facing_bar(center, facing_idx)
                                + '</g>'
                            )

            # ── Purchase: ghost piece + cost ring at target hex ──
            if move.purchase is not None:
                loc  = move.purchase.location
                flag = None
                if squad is not None:
                    flag = squad.flag
                elif ctx.squads:
                    flag = ctx.squads[0].flag

                if flag is not None and loc.parent is not None:
                    coarse_idx = loc.parent.hp2i(loc.location)
                    for fi in ctx.fine_indices(coarse_idx):
                        if 0 <= fi < N:
                            center = grid.hexes[fi].center
                            pid    = f"mv_buy_{move.purchase.piece_type.value}"
                            parts.append(
                                flag.draw_piece(
                                    move.purchase.piece_type, center, ctx.builder,
                                    scale=scale, size='board',
                                    piece_id=pid, opacity=opacity,
                                )
                            )
                            # Cost ring: dashed circle around the hex
                            cost = PURCHASE_COST.get(move.purchase.piece_type, 0)
                            r    = grid.radius * 0.75
                            parts.append(
                                f'<circle cx="{center.x:.1f}" cy="{center.y:.1f}" '
                                f'r="{r:.1f}" fill="none" '
                                f'stroke="{flag.darkPrimary}" stroke-width="2" '
                                f'stroke-dasharray="4 3" opacity="{opacity}"/>'
                            )
                            # Cost label
                            parts.append(
                                f'<text x="{center.x:.1f}" y="{center.y - r - 4:.1f}" '
                                f'text-anchor="middle" font-size="10" '
                                f'fill="{flag.darkPrimary}" opacity="{opacity}" '
                                f'font-weight="bold">-{cost}🍖</text>'
                            )

        return '\n'.join(parts)

    return OverlaySpec("move_ghost", render, requires={'pieces'}, priority=85)
    # priority 85 = just above PieceOverlay (80), below SquadNamesOverlay (90)
```

Key design choices:
- **Rotations**: reuses `DiagramGlyphs.rot_arrow` + `facing_bar` exactly as `LanternOverlay` does — no new SVG code
- **Placements**: uses `flag.draw_piece(..., opacity=opacity)` — identical to `PieceOverlay` but at the target hex instead of `piece.place`
- **Purchases**: same ghost piece + a dashed ring (cheap SVG, no new primitives) + cost label
- **Priority 85** slots it cleanly between `PieceOverlay` (80) and `SquadNamesOverlay` (90)
- `squad` is an optional closure param — if `None` it falls back to `ctx.squads[0]`, same pattern as `move_ft`

In [ ]:
#| export
def MoveOverlay(
    moves: list['Move'],
    squad: 'Squad' = None,
    opacity: float = 0.45,
    show_rotation_arrows: bool = True,
    **kw
) -> OverlaySpec:
    """Ghost-render proposed move state: placements, purchases, rotation arrows.
    
    Sits above PieceOverlay in the stack. Uses draw_piece (with opacity) for
    ghost pieces, DiagramGlyphs.rot_arrow + facing_bar for rotations.
    """
    def render(ctx) -> str:
        grid   = ctx.grid
        N      = len(grid.hexes)
        scale  = grid.radius * 0.8 / 22.5
        parts  = []

        for move in moves:
            # ── Rotations: rot_arrow + new facing_bar at piece's current hex ──
            if show_rotation_arrows:
                for r in move.rotations:
                    if r.piece.place is None or r.piece.flag is None:
                        continue
                    coarse_idx = r.piece.place.parent.hp2i(r.piece.place.location)
                    for fi in ctx.fine_indices(coarse_idx):
                        if 0 <= fi < N:
                            c     = grid.hexes[fi].center
                            size  = grid.radius
                            glyphs = DiagramGlyphs(r.piece.flag, size=size)
                            glyphs.register_styles(ctx.builder)

                            dirs       = HexPosition.directions()
                            old_facing = dirs.index(r.piece.place.facing) if r.piece.place.facing in dirs else 0
                            new_facing = dirs.index(r.facing)             if r.facing             in dirs else 0

                            # Ghost of the new facing bar
                            parts.append(
                                f'<g opacity="{opacity}">'
                                + glyphs.facing_bar(c, new_facing)
                                + '</g>'
                            )
                            # Rotation arrow (clockwise if turning right)
                            clockwise = (new_facing - old_facing) % 6 <= 3
                            parts.append(
                                f'<g opacity="{opacity}">'
                                + glyphs.rot_arrow(c, r=size * 0.55, clockwise=clockwise)
                                + '</g>'
                            )

            # ── Placement: ghost piece at target hex ──
            if move.placement is not None:
                p   = move.placement.piece
                loc = move.placement.location
                if p.flag is not None and loc.parent is not None:
                    coarse_idx = loc.parent.hp2i(loc.location)
                    for fi in ctx.fine_indices(coarse_idx):
                        if 0 <= fi < N:
                            center = grid.hexes[fi].center
                            pid    = f"mv_place_{p.id}"
                            parts.append(
                                p.flag.draw_piece(
                                    p.type, center, ctx.builder,
                                    scale=scale, size='board',
                                    piece_id=pid, opacity=opacity,
                                )
                            )
                            # Facing bar for initial facing
                            glyphs = DiagramGlyphs(p.flag, size=grid.radius)
                            glyphs.register_styles(ctx.builder)
                            dirs       = HexPosition.directions()
                            facing_idx = dirs.index(loc.facing) if loc.facing in dirs else 0
                            parts.append(
                                f'<g opacity="{opacity}">'
                                + glyphs.facing_bar(center, facing_idx)
                                + '</g>'
                            )

            # ── Purchase: ghost piece + cost ring at target hex ──
            if move.purchase is not None:
                loc  = move.purchase.location
                flag = None
                if squad is not None:
                    flag = squad.flag
                elif ctx.squads:
                    flag = ctx.squads[0].flag

                if flag is not None and loc.parent is not None:
                    coarse_idx = loc.parent.hp2i(loc.location)
                    for fi in ctx.fine_indices(coarse_idx):
                        if 0 <= fi < N:
                            center = grid.hexes[fi].center
                            pid    = f"mv_buy_{move.purchase.piece_type.value}"
                            parts.append(
                                flag.draw_piece(
                                    move.purchase.piece_type, center, ctx.builder,
                                    scale=scale, size='board',
                                    piece_id=pid, opacity=opacity,
                                )
                            )
                            # Cost ring: dashed circle around the hex
                            cost = PURCHASE_COST.get(move.purchase.piece_type, 0)
                            r    = grid.radius * 0.75
                            parts.append(
                                f'<circle cx="{center.x:.1f}" cy="{center.y:.1f}" '
                                f'r="{r:.1f}" fill="none" '
                                f'stroke="{flag.darkPrimary}" stroke-width="2" '
                                f'stroke-dasharray="4 3" opacity="{opacity}"/>'
                            )
                            # Cost label
                            parts.append(
                                f'<text x="{center.x:.1f}" y="{center.y - r - 4:.1f}" '
                                f'text-anchor="middle" font-size="10" '
                                f'fill="{flag.darkPrimary}" opacity="{opacity}" '
                                f'font-weight="bold">-{cost}🍖</text>'
                            )

        return '\n'.join(parts)

    return OverlaySpec("move_ghost", render, requires={'pieces'}, priority=85)
    # priority 85 = just above PieceOverlay (80), below SquadNamesOverlay (90)


## Move Log

In [ ]:
#| export
@dataclass
class MoveLog:
    parts: GameParts
    _df: pd.DataFrame = field(default_factory=lambda: pd.DataFrame(
        columns=['turn', 'squad', 'piece', 'action', 'detail']))
    _moves: list[tuple[int, Squad, Move]] = field(default_factory=list)
    _turn: int = 0  # current turn counter

    def next_turn(self):
        """Advance to the next turn."""
        self._turn += 1

    def record(self, squad: Squad, move: Move, turn: int = None):
        """Validate, apply, and log."""
        validated = Move.from_squad(squad, move.rotations, move.placement, move.purchase)
        validated.apply(self.parts, squad)
        
        t = turn if turn is not None else self._turn
        self._moves.append((t, squad, validated))

        rows = []
        for r in validated.rotations:
            rows.append(dict(turn=t, squad=squad.name, piece=r.piece.name,
                             action='rotate', detail=r.facing.label))
        if validated.placement:
            p = validated.placement
            idx = self.parts.hp2i(p.location.location)
            rows.append(dict(turn=t, squad=squad.name, piece=p.piece.name,
                             action='place', detail=str(idx)))
        if validated.purchase:
            rows.append(dict(turn=t, squad=squad.name,
                             piece=validated.purchase.piece_type.value,
                             action='purchase',
                             detail=str(self.parts.hp2i(validated.purchase.location.location))))
        self._df = pd.concat([self._df, pd.DataFrame(rows)], ignore_index=True)

    @property
    def df(self) -> pd.DataFrame: return self._df
    def for_squad(self, name: str) -> pd.DataFrame:
        return self._df[self._df.squad == name]
    def turn_count(self) -> int:
        return int(self._df.turn.max() + 1) if len(self._df) else 0
    def move_for_turn(self, turn: int) -> tuple[Squad, Move] | None:
        for t, squad, move in self._moves:
            if t == turn: return squad, move
        return None


In [ ]:
#| export
@patch
def moveLogDisplay(log: MoveLog) -> FT:
    """Scrollable move log: compact table rows that expand to move_ft cards."""
    if not log._moves:
        return Div(P("No moves recorded yet.", cls=TextPresets.muted_sm), id="move-log")

    action_icons = {'rotate': '↻', 'place': '📍', 'purchase': '🛒'}

    rows = []
    for turn, squad, move in reversed(log._moves):   # newest first
        # Summary chips for this move
        actions = log.df[log.df.turn == turn]
        chips = []
        for _, row in actions.iterrows():
            icon = action_icons.get(row.action, '·')
            chips.append(Span(
                f"{icon} {row.piece}",
                style=f"background:{squad.flag.lightComp};"
                      f"color:{squad.flag.darkPrimary};"
                      f"border-radius:6px;padding:2px 7px;font-size:11px;font-weight:600;"
            ))

        # The expandable detail — move_ft rendered inline on toggle
        detail_id = f"move-detail-{turn}"
        move_card = log.parts.move_ft(move, squad=squad)

        row = Div(
            # Summary row (always visible)
            Div(
                # Turn badge
                Div(
                    Div(f"T{turn}", style=f"font-size:10px;font-weight:800;"
                        f"color:{squad.flag.darkPrimary};"),
                    style=f"background:{squad.flag.lightComp};"
                          f"border-radius:8px;padding:4px 8px;min-width:32px;text-align:center;"
                ),
                # Squad logo + name
                Div(
                    NotStr(squad.flag.logo(size=24, piece_id=f"log_{turn}")),
                    Span(squad.name, style="font-size:11px;font-weight:600;"),
                    style="display:flex;align-items:center;gap:6px;"
                ),
                # Action chips
                Div(*chips, style="display:flex;flex-wrap:wrap;gap:4px;flex:1;"),
                # Expand toggle
                Button("▾", 
                    style="font-size:16px;background:none;border:none;cursor:pointer;",
                    onclick=f"document.getElementById('{detail_id}').classList.toggle('hidden')"
                ),
                style="display:flex;align-items:center;gap:10px;padding:8px 4px;"
                      "border-bottom:1px solid #f0f0f0;cursor:default;"
            ),
            # Expandable move card (hidden by default)
            Div(move_card, id=detail_id, cls="hidden",
                style="padding:8px 0;"),
        )
        rows.append(row)

    return Div(
        DivFullySpaced(
            H4("Move Log", cls="font-bold text-base"),
            Small(f"{log.turn_count()} turns · {len(log._moves)} moves",
                  cls=TextPresets.muted_sm),
        ),
        Div(*rows,
            style="max-height:480px;overflow-y:auto;border:1px solid #e5e5e5;"
                  "border-radius:12px;padding:4px 8px;"),
        id="move-log",
    )


In [ ]:
#| export
def PlacementInfo(pending: PendingMove, parts: GameParts = None) -> FT:
    """Live preview of the pending move via move_ft."""
    # Try to extract GameParts from any placed piece or pending placement
    if parts is None:
        if pending.placement and pending.placement.location.parent:
            parts = pending.placement.location.parent
        elif pending.squad:
            for p in pending.squad.pieces:
                if p.place and p.place.parent:
                    parts = p.place.parent
                    break

    # Status message for incomplete actions
    if pending.selected_piece is not None and pending.placement is None:
        hint = P(f"Click a hex to place {pending.selected_piece.name}.",
                 cls=TextPresets.muted_sm)
    elif pending.purchase_type is not None and pending.purchase is None:
        hint = P(f"Click a hex for your new {pending.purchase_type.value.title()}.",
                 cls=TextPresets.muted_sm)
    else:
        hint = ""

    # Build a preview Move from whatever's been set so far
    if pending.has_actions and parts is not None:
        preview = Move(
            rotations=list(pending.rotations),
            placement=pending.placement,
            purchase=pending.purchase,
        )
        card = parts.move_ft(preview, squad=pending.squad)
    else:
        card = P("Select a piece to place, or choose a purchase above.",
                 cls=TextPresets.muted_sm)

    return Div(
        H4("Move Preview", cls="font-bold text-base mb-2"),
        hint,
        card,
        id="placement-info",
    )


In [ ]:
#| export


def RotationPanel(pending: PendingMove) -> FT:
    """Shows pending rotations for all placed pieces in the squad."""
    if pending.squad is None:
        return Div(P("No squad selected.", cls=TextPresets.muted_sm), id="rotation-panel")

    placed = [p for p in pending.squad.pieces if p.place is not None]
    pending_map = {r.piece: r for r in pending.rotations}

    rows = []
    for piece in placed:
        rot = pending_map.get(piece)
        facing_label = rot.facing.label if rot else piece.place.facing.label
        is_pending = rot is not None

        rows.append(DivFullySpaced(
            DivLAligned(
                NotStr(piece.flag.pieceLogo(piece.type, size=32)),
                DivVStacked(
                    Strong(piece.name, cls="text-sm"),
                    Small(f"facing: {facing_label}"
                          + (" ← new" if is_pending else ""),
                          cls=TextPresets.muted_sm),
                ), cls="gap-2"
            ),
            Button(
                "↩ undo" if is_pending else "↻ rotate",
                cls=(ButtonT.primary if is_pending else ButtonT.ghost) + " text-xs",
                hx_post="/move/toggle_rotation",
                hx_vals=f'{{"piece_id": "{piece.id}"}}',
                hx_target="#move-panel", hx_swap="outerHTML",
            ),
        ))

    return Div(
        H4("Rotations", cls="font-bold text-base mb-2"),
        DivVStacked(*rows, cls="gap-2") if rows
            else P("No pieces on board yet.", cls=TextPresets.muted_sm),
        id="rotation-panel",
    )




In [ ]:
#| export
@patch
def movePanel(self:GameParts, pending: PendingMove) -> FT:
    """Full move-building side panel driven entirely by PendingMove state."""
    if pending.squad is None:
        return Card(P("No squad selected.", cls=TextPresets.muted_sm),
                    id="move-panel", cls="p-4 w-72")

    flag = pending.squad.flag

    # Action count for commit button label
    n = (len(pending.rotations)
         + (1 if pending.placement else 0)
         + (1 if pending.purchase else 0))

    return Card(
        DivFullySpaced(
            Button("Clear", cls=ButtonT.ghost,
                   hx_post="/move/clear",
                   hx_target="#move-panel", hx_swap="outerHTML"),
            Button(f"Commit Move ({n})" if n else "Nothing to commit",
                   cls=ButtonT.primary,
                   disabled=not pending.has_actions,
                   hx_post="/move/commit",
                   hx_target="#move-panel", hx_swap="outerHTML"),
        ),
        Divider(),
        PurchasePanel( pending),
        Divider(),
        PlacementInfo(pending,parts=self),
        Divider(),
        RotationPanel(pending),
        
        id="move-panel",
        cls="p-4 space-y-3 w-72",
    )


I think PlacementInfo could be refactored to use move_ft

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good call — `move_ft` already renders placement/rotation/purchase cards, so `PlacementInfo` is duplicating work with less fidelity. The main thing is getting a `GameParts` reference. We can pull it from the pending state itself:

```python
def PlacementInfo(pending: PendingMove, parts: GameParts = None) -> FT:
    """Live preview of the pending move via move_ft."""
    # Try to extract GameParts from any placed piece or pending placement
    if parts is None:
        if pending.placement and pending.placement.location.parent:
            parts = pending.placement.location.parent
        elif pending.squad:
            for p in pending.squad.pieces:
                if p.place and p.place.parent:
                    parts = p.place.parent
                    break

    # Status message for incomplete actions
    if pending.selected_piece is not None and pending.placement is None:
        hint = P(f"Click a hex to place {pending.selected_piece.name}.",
                 cls=TextPresets.muted_sm)
    elif pending.purchase_type is not None and pending.purchase is None:
        hint = P(f"Click a hex for your new {pending.purchase_type.value.title()}.",
                 cls=TextPresets.muted_sm)
    else:
        hint = ""

    # Build a preview Move from whatever's been set so far
    if pending.has_actions and parts is not None:
        preview = Move(
            rotations=list(pending.rotations),
            placement=pending.placement,
            purchase=pending.purchase,
        )
        card = parts.move_ft(preview, squad=pending.squad)
    else:
        card = P("Select a piece to place, or choose a purchase above.",
                 cls=TextPresets.muted_sm)

    return Div(
        H4("Move Preview", cls="font-bold text-base mb-2"),
        hint,
        card,
        id="placement-info",
    )
```

Changes from the original:
- **No manual string building** — delegates the entire visual to `move_ft`, which already handles all three action types with map thumbnails
- **Live preview** — as the user adds rotations/placements/purchases to `PendingMove`, the card updates to show exactly what will be committed
- **Hint text** preserved for the "waiting for hex click" states where placement/purchase is incomplete
- `parts` auto-detected from the pending state so callers don't always need to pass it

You'll also want to update `MovePanel` to pass `parts` through if you have it handy, though the auto-detection should cover most cases.

In [ ]:
#| export
def PurchasePreview(flag: CountryFlag, piece_type: PieceType, 
                    year: int = 1900) -> FT:
    """Preview card for a piece type being considered for purchase."""
    # Build a throwaway unplaced Piece just for rendering
    rng = random.Random(hash((flag.name, piece_type.value)))
    names = flag.commonNames(year=year)
    name = rng.choice(names)
    
    preview = Piece(
        id=f"preview_{piece_type.value}",
        flag=flag,
        type=piece_type,
        name=name,
        year=year + rng.randint(20, 38),
        animal=flag.animal_name(),
        countryName=flag.countryPrefix,
    )
    preview.lantern = StyleCSS(f"preview_lantern", fill=flag.primary, opacity=0.3)
    
    return Div(
        H4("Preview", cls="font-bold text-base mb-2"),
        preview.__ft__(),
        id="purchase-preview",
    )


Some of this should be refactored to use PendingMove

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good call — `PendingMove` already holds all the transient UI state that's being passed around as loose params. Here's the refactored set:

```python
def PurchasePanel(flag: CountryFlag, pending: PendingMove,
                  hx_target: str = "#move-panel") -> FT:
    """Vertical list of purchasable pieces with costs and affordability."""
    # Find king pantry
    king_pantry = 0
    if pending.squad:
        kings = [p for p in pending.squad.pieces
                 if p.type == PieceType.KING and p.place is not None]
        if kings:
            king_pantry = kings[0].place.pantry

    rows = []
    for pt, cost in PURCHASE_COST.items():
        affordable = king_pantry >= cost
        is_selected = pt == pending.purchase_type

        logo_svg = NotStr(flag.pieceLogo(pt, size=48))
        cost_badge = Span(
            f"🌾 {cost}",
            cls=f"text-sm font-bold {'text-green-600' if affordable else 'text-red-400'}"
        )
        pantry_note = (Small("affordable", cls="text-xs text-green-500") if affordable
                       else Small(f"pantry: {king_pantry}", cls=TextPresets.muted_sm))

        label = DivVStacked(Strong(pt.value.title(), cls="text-sm"), cost_badge, pantry_note)

        btn = Button(
            DivLAligned(logo_svg, label, cls="gap-3"),
            disabled=not affordable,
            cls=(f"w-full text-left rounded-xl p-2 transition-all "
                 + ("ring-2 ring-offset-1 " if is_selected else "")
                 + ("opacity-40 cursor-not-allowed" if not affordable
                    else "hover:bg-base-200 cursor-pointer")),
            hx_post="/move/select_purchase",
            hx_vals=f'{{"piece_type": "{pt.value}"}}',
            hx_target=hx_target, hx_swap="outerHTML",
            **({"aria_pressed": "true"} if is_selected else {}),
        )
        rows.append(btn)

    return Div(
        H4("Purchase", cls="font-bold text-base mb-2"),
        P(f"King's pantry: 🌾 {king_pantry}", cls=TextPresets.muted_sm),
        DivVStacked(*rows, cls="gap-1"),
        id="purchase-panel",
    )


def PlacementInfo(pending: PendingMove) -> FT:
    """Shows placement state: which piece is selected, where it's going."""
    if pending.selected_piece is not None:
        p = pending.selected_piece
        if pending.placement is not None:
            idx = pending.placement.location.parent.hp2i(pending.placement.location.location)
            msg = f"Placing {p.name} ({p.type.value.title()}) at hex {idx}."
        else:
            msg = f"Click a highlighted hex to place {p.name}."
    elif pending.purchase_type is not None:
        msg = f"Click a hex to place your new {pending.purchase_type.value.title()}."
    else:
        msg = "Select a piece to place, or choose a purchase above."

    return Div(
        H4("Placement", cls="font-bold text-base mb-2"),
        P(msg, cls=TextPresets.muted_sm),
        id="placement-info",
    )


def RotationPanel(pending: PendingMove) -> FT:
    """Shows pending rotations for all placed pieces in the squad."""
    if pending.squad is None:
        return Div(P("No squad selected.", cls=TextPresets.muted_sm), id="rotation-panel")

    placed = [p for p in pending.squad.pieces if p.place is not None]
    pending_map = {r.piece: r for r in pending.rotations}

    rows = []
    for piece in placed:
        rot = pending_map.get(piece)
        facing_label = rot.facing.label if rot else piece.place.facing.label
        is_pending = rot is not None

        rows.append(DivFullySpaced(
            DivLAligned(
                NotStr(piece.flag.pieceLogo(piece.type, size=32)),
                DivVStacked(
                    Strong(piece.name, cls="text-sm"),
                    Small(f"facing: {facing_label}"
                          + (" ← new" if is_pending else ""),
                          cls=TextPresets.muted_sm),
                ), cls="gap-2"
            ),
            Button(
                "↩ undo" if is_pending else "↻ rotate",
                cls=(ButtonT.primary if is_pending else ButtonT.ghost) + " text-xs",
                hx_post="/move/toggle_rotation",
                hx_vals=f'{{"piece_id": "{piece.id}"}}',
                hx_target="#move-panel", hx_swap="outerHTML",
            ),
        ))

    return Div(
        H4("Rotations", cls="font-bold text-base mb-2"),
        DivVStacked(*rows, cls="gap-2") if rows
            else P("No pieces on board yet.", cls=TextPresets.muted_sm),
        id="rotation-panel",
    )


def MovePanel(pending: PendingMove) -> FT:
    """Full move-building side panel driven entirely by PendingMove state."""
    if pending.squad is None:
        return Card(P("No squad selected.", cls=TextPresets.muted_sm),
                    id="move-panel", cls="p-4 w-72")

    flag = pending.squad.flag

    # Action count for commit button label
    n = (len(pending.rotations)
         + (1 if pending.placement else 0)
         + (1 if pending.purchase else 0))

    return Card(
        PurchasePanel(flag, pending),
        Divider(),
        PlacementInfo(pending),
        Divider(),
        RotationPanel(pending),
        Divider(),
        DivFullySpaced(
            Button("Clear", cls=ButtonT.ghost,
                   hx_post="/move/clear",
                   hx_target="#move-panel", hx_swap="outerHTML"),
            Button(f"Commit Move ({n})" if n else "Nothing to commit",
                   cls=ButtonT.primary,
                   disabled=not pending.has_actions,
                   hx_post="/move/commit",
                   hx_target="#move-panel", hx_swap="outerHTML"),
        ),
        id="move-panel",
        cls="p-4 space-y-3 w-72",
    )
```

What changed:
- **`MovePanel`** now takes a single `PendingMove` — no more `king_pantry`, `selected_purchase_type`, `target_hex`, `pending_rotations` scattered params
- **`PurchasePanel`** derives `king_pantry` from `pending.squad` internally
- **`PlacementInfo`** reads both `pending.selected_piece` and `pending.purchase_type` for richer status messages
- **`RotationPanel`** builds its rotation map from `pending.rotations` directly, and the button is now a toggle (`↻ rotate` / `↩ undo`) matching the `add_rotation` / `remove_rotation` API
- **Commit button** shows action count and disables when `not pending.has_actions`
- The HTMX endpoint names stay the same — server handlers just mutate the shared `PendingMove` instance and re-render `MovePanel(pending)`

In [ ]:
myStuff = GameParts()

In [ ]:
myStuff.loadSquads(year=1955)

In [ ]:
ourTroop = myStuff.squads[0]
#show(ourTroop.pieces[0])

#| export
@dataclass
class PendingMove:
    """Mutable accumulator for a move being built by the player."""
    squad: Squad | None = None
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None
    purchase: Purchase | None = None

    # ── UI-only transient state ──
    purchase_type: PieceType | None = None   # selected but not yet placed on map
    selected_piece: Piece | None = None      # piece picked for placement (awaiting hex click)

    def commit(self) -> Move:
        """Validate and produce a committed Move. Raises MoveError."""
        if self.squad is None:
            raise MoveError("no squad assigned")
        return Move.from_squad(self.squad,
            rotations=self.rotations,
            placement=self.placement,
            purchase=self.purchase)

    def clear(self):
        """Reset all pending state."""
        self.rotations.clear()
        self.placement = None
        self.purchase = None
        self.purchase_type = None
        self.selected_piece = None

    @property
    def has_actions(self) -> bool:
        return bool(self.rotations or self.placement or self.purchase)

    # ── Convenience builders ──
    def add_rotation(self, piece: Piece, facing: HexPosition):
        # Replace if already rotating this piece
        self.rotations = [r for r in self.rotations if r.piece != piece]
        self.rotations.append(Rotation(piece, facing))

    def remove_rotation(self, piece: Piece):
        self.rotations = [r for r in self.rotations if r.piece != piece]

    def set_placement(self, piece: Piece, location: 'MapPlace'):
        self.placement = Placement(piece, location)
        self.selected_piece = None

    def set_purchase(self, piece_type: PieceType, location: 'MapPlace', facing: HexPosition):
        self.purchase = Purchase(piece_type, location, facing)
        self.purchase_type = None


In [ ]:
#show(ourTroop)

In [ ]:
#| export
def piece_logo_html(flag: CountryFlag, pt: PieceType, size: int = 56) -> FT:
    """Inline-safe piece logo with XML declaration stripped."""
    raw = flag.pieceLogo(pt, size=size)
    raw = raw.replace("<?xml version='1.0' encoding='utf-8'?>", "").strip()
    return Div(NotStr(raw), style=f"width:{size}px;height:{size}px;flex-shrink:0;")


def PurchasePanel(pending: PendingMove,
                  hx_target: str = "#move-panel") -> FT:
    """Vertical list of purchasable pieces with costs and affordability."""
    flag = pending.squad.flag
    king_pantry = 0
    if pending.squad:
        kings = [p for p in pending.squad.pieces
                 if p.type == PieceType.KING and p.place is not None]
        if kings:
            king_pantry = kings[0].place.pantry

    rows = []
    for pt, cost in PURCHASE_COST.items():
        affordable = king_pantry >= cost
        is_selected = pt == pending.purchase_type

        cost_badge = Span(
            f"🍖 {cost}",
            style="font-size:1.1rem;",
            cls=f"font-bold {'text-green-600' if affordable else 'text-red-400'}"
        )
        pantry_note = Small(
            f"need {cost - king_pantry} more",
            cls=TextPresets.muted_sm
        ) if not affordable else Small("✓ affordable", cls="text-xs text-green-500")

        label = DivVStacked(
            Strong(pt.value.title(), cls="text-sm"),
            cost_badge,
            pantry_note,
        )

        btn = Button(
            DivLAligned(piece_logo_html(flag, pt, size=56), label, cls="gap-3"),
            disabled=not affordable,
            cls=(
                "w-full text-left rounded-xl p-2 transition-all "
                + ("ring-2 ring-offset-1 ring-primary " if is_selected else "")
                + ("opacity-40 cursor-not-allowed" if not affordable 
                   else "hover:bg-base-200 cursor-pointer")
            ),
            hx_post="/move/select_purchase",
            hx_vals=f'{{"piece_type": "{pt.value}"}}',
            hx_target=hx_target,
            hx_swap="outerHTML",
            **({"aria_pressed": "true"} if is_selected else {}),
        )
        rows.append(btn)

    return Div(
        DivFullySpaced(
            H4("Purchase", cls="font-bold text-base"),
            Span(f"🧺 {king_pantry}", style="font-size:1.2rem;font-weight:700;"),
        ),
        DivVStacked(*rows, cls="gap-1 mt-2"),
        id="purchase-panel",
    )


Should PurchasePanel be refactored to use PendingMove?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — `PendingMove` already has everything `PurchasePanel` needs:

- `pending.squad.flag` → `flag`
- `pending.purchase_type` → `selected_type`
- `king_pantry` is derivable from `pending.squad.pieces`

So the signature collapses to:

```python
def PurchasePanel(pending: PendingMove,
                  hx_target: str = "#move-panel") -> FT:
    """Vertical list of purchasable pieces with costs and affordability."""
    flag = pending.squad.flag
    king_pantry = 0
    if pending.squad:
        kings = [p for p in pending.squad.pieces
                 if p.type == PieceType.KING and p.place is not None]
        if kings:
            king_pantry = kings[0].place.pantry

    rows = []
    for pt, cost in PURCHASE_COST.items():
        affordable = king_pantry >= cost
        is_selected = pt == pending.purchase_type

        cost_badge = Span(
            f"🍖 {cost}",
            style="font-size:1.1rem;",
            cls=f"font-bold {'text-green-600' if affordable else 'text-red-400'}"
        )
        pantry_note = Small(
            f"need {cost - king_pantry} more",
            cls=TextPresets.muted_sm
        ) if not affordable else Small("✓ affordable", cls="text-xs text-green-500")

        label = DivVStacked(
            Strong(pt.value.title(), cls="text-sm"),
            cost_badge,
            pantry_note,
        )

        btn = Button(
            DivLAligned(piece_logo_html(flag, pt, size=56), label, cls="gap-3"),
            disabled=not affordable,
            cls=(
                "w-full text-left rounded-xl p-2 transition-all "
                + ("ring-2 ring-offset-1 ring-primary " if is_selected else "")
                + ("opacity-40 cursor-not-allowed" if not affordable 
                   else "hover:bg-base-200 cursor-pointer")
            ),
            hx_post="/move/select_purchase",
            hx_vals=f'{{"piece_type": "{pt.value}"}}',
            hx_target=hx_target,
            hx_swap="outerHTML",
            **({"aria_pressed": "true"} if is_selected else {}),
        )
        rows.append(btn)

    return Div(
        DivFullySpaced(
            H4("Purchase", cls="font-bold text-base"),
            Span(f"🧺 {king_pantry}", style="font-size:1.2rem;font-weight:700;"),
        ),
        DivVStacked(*rows, cls="gap-1 mt-2"),
        id="purchase-panel",
    )
```

This way every panel in `MovePanel` speaks the same language — one `PendingMove` flows through all of them, and callers never need to remember which loose params to fish out.

In [ ]:
pending = PendingMove(
    squad=ourTroop,
    purchase_type=PieceType.BISHOP,   # one selected in purchase panel
    selected_piece=ourTroop.pieces[0], # piece awaiting hex click
)
myStuff.movePanel(pending)


What went wrong?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`MovePanel` still has the old call signature — it's passing `flag` as the first argument:

```python
PurchasePanel(flag, pending)  # ← old call
```

But `PurchasePanel` was refactored to take just `pending`. So `flag` (a `CountryFlag`) is landing in the `pending` parameter, hence `'CountryFlag' object has no attribute 'squad'`.

Fix is one line in `MovePanel`:

```python
PurchasePanel(pending),  # ← drop flag
```

In [ ]:
#| export

@patch
def purchases(self: MoveLog) -> pd.DataFrame:
    return self._df[self._df.action == 'purchase']


In [ ]:
#| export
@patch
def purchase_summary(self: MoveLog) -> pd.DataFrame:
    df = self._df[self._df.action == 'purchase'].copy()
    df = df.rename(columns={'piece': 'piece_type', 'detail': 'hex_idx'})
    df['hex_idx'] = df['hex_idx'].astype(int)
    df['cost'] = df['piece_type'].map({pt.value: cost for pt, cost in PURCHASE_COST.items()})
    return df[['turn', 'squad', 'piece_type', 'hex_idx', 'cost']]


@patch
def _find_purchased_piece(squad: Squad, parts: GameParts, 
                           piece_type: PieceType, hex_idx: int) -> 'Piece | None':
    """Find a placed piece in squad matching type and hex index."""
    for p in squad.pieces:
        if (p.type == piece_type 
                and p.place is not None 
                and parts.hp2i(p.place.location) == hex_idx):
            return p
    return None


@patch
def PurchaseTable(log: MoveLog) -> FT:
    df = log.purchase_summary()
    if df.empty:
        return Div(P("No purchases yet.", cls=TextPresets.muted_sm))

    rows = []
    for _, row in df.iterrows():
        result = log.move_for_turn(int(row.turn))
        if result is None: continue
        squad, move = result

        pt = PieceType(row.piece_type)
        piece = squad._find_purchased_piece( log.parts, pt, int(row.hex_idx))

        detail_id = f"purchase-detail-{row.turn}"
        detail = piece.__ft__() if piece is not None else P("Piece no longer on board.", cls=TextPresets.muted_sm)

        rows.append(Tr(
            Td(f"T{row.turn}", cls="font-mono text-sm"),
            Td(DivLAligned(
                NotStr(squad.flag.logo(size=24, piece_id=f"pur_{row.turn}")),
                Span(row.squad, cls="text-sm"),
                cls="gap-2"
            )),
            Td(NotStr(squad.flag.pieceLogo(pt, size=32))),
            Td(row.piece_type.title(), cls="text-sm font-medium"),
            Td(f"🍖 {row.cost}", cls="text-sm font-bold"),
            Td(f"hex {row.hex_idx}", cls=TextPresets.muted_sm),
            Td(Button("▾",
                style="background:none;border:none;cursor:pointer;",
                onclick=f"document.getElementById('{detail_id}').classList.toggle('hidden')"
            )),
        ))
        rows.append(Tr(
            Td(Div(detail, id=detail_id, cls="hidden"), colspan="7")
        ))

    return Div(
        H4("Purchases", cls="font-bold text-base mb-2"),
        Table(
            Thead(Tr(Th("Turn"), Th("Squad"), Th(""), Th("Type"),
                     Th("Cost"), Th("Hex"), Th(""))),
            Tbody(*rows),
        ),
    )


@patch
def PurchaseTable(log: MoveLog) -> FT:
    df = log.purchase_summary()
    if df.empty:
        return Div(P("No purchases yet.", cls=TextPresets.muted_sm))

    squads_by_name = {sq.name: sq for sq in log.parts.squads}
    all_squads = list(squads_by_name.keys())
    all_turns  = sorted(df.turn.unique())

    # ── Header: one column per squad ──
    squad_headers = []
    for name in all_squads:
        sq = squads_by_name[name]
        squad_headers.append(Th(
            DivVStacked(
                NotStr(sq.flag.logo(size=32, piece_id=f"pt_hdr_{name}")),
                Span(name, style=f"font-size:10px;font-weight:700;"
                                 f"color:{sq.flag.darkPrimary};text-align:center;"),
            ),
            style=f"background:{sq.flag.lightComp};padding:8px;min-width:120px;"
        ))

    # ── One row per turn ──
    body_rows = []
    for turn in all_turns:
        turn_purchases = df[df.turn == turn]
        cells = [Td(
            Span(f"T{turn}", style="font-size:11px;font-weight:800;font-family:monospace;"),
            style="vertical-align:top;padding:6px;border-right:1px solid #eee;"
        )]

        for name in all_squads:
            sq   = squads_by_name[name]
            rows = turn_purchases[turn_purchases.squad == name]

            if rows.empty:
                cells.append(Td(
                    Div("—", style="color:#ddd;text-align:center;padding:16px;"),
                    style=f"background:{sq.flag.lightComp}08;vertical-align:top;"
                ))
                continue

            # Could be multiple purchases in one turn
            cards = []
            for _, row in rows.iterrows():
                pt    = PieceType(row.piece_type)
                piece = squad._find_purchased_piece(log.parts, pt, int(row.hex_idx))
                if piece is not None:
                    cards.append(piece.__ft__())
                else:
                    cards.append(DivVStacked(
                        NotStr(sq.flag.pieceLogo(pt, size=40)),
                        Span(row.piece_type.title(),
                             style="font-size:11px;font-weight:600;"),
                        Span(f"hex {row.hex_idx} · 🍖{row.cost}",
                             style="font-size:10px;color:#888;"),
                    ))

            cells.append(Td(
                *cards,
                style=f"vertical-align:top;padding:4px;"
                      f"border-left:2px solid {sq.flag.lightComp};"
            ))

        body_rows.append(Tr(*cells,
            style="border-bottom:1px solid #f0f0f0;"
        ))

    return Div(
        DivFullySpaced(
            H4("Purchases", cls="font-bold text-base"),
            Small(f"{len(df)} total · {len(all_turns)} turns",
                  cls=TextPresets.muted_sm),
        ),
        Div(
            Table(
                Thead(Tr(Th("Turn"), *squad_headers)),
                Tbody(*body_rows),
            ),
            style="overflow-x:auto;border:1px solid #e5e5e5;border-radius:12px;"
        ),
    )


def _piece_mini(piece: Piece, parts: GameParts) -> FT:
    """Compact piece cell: logo + name + hex index."""
    idx = parts.hp2i(piece.place.location) if piece.place else "?"
    return DivVStacked(
        NotStr(piece.flag.pieceLogo(piece.type, size=40)),
        Span(piece.name, style="font-size:11px;font-weight:700;text-align:center;"),
        Span(f"hex {idx}", style="font-size:10px;color:#888;"),
        style="align-items:center;gap:2px;padding:4px;"
    )
@patch
def PurchaseTable(log: MoveLog) -> FT:
    df = log.purchase_summary()
    if df.empty:
        return Div(P("No purchases yet.", cls=TextPresets.muted_sm))

    squads_by_name = {sq.name: sq for sq in log.parts.squads}
    all_squads     = list(squads_by_name.keys())
    all_turns      = sorted(df.turn.unique())

    # ── Header row ──
    header = Tr(
        Th("Turn", style="padding:8px;font-size:11px;text-align:center;"),
        *[
            Th(
                DivVStacked(
                    NotStr(squads_by_name[name].flag.logo(
                        size=28, piece_id=f"ph_{name}")),
                    Span(name, style=f"font-size:10px;font-weight:700;"
                                     f"color:{squads_by_name[name].flag.darkPrimary};"
                                     f"text-align:center;max-width:80px;"),
                    style="align-items:center;gap:3px;"
                ),
                style=f"padding:8px;min-width:90px;text-align:center;"
                      f"background:{squads_by_name[name].flag.lightComp};"
                      f"border-bottom:2px solid {squads_by_name[name].flag.primary};"
            )
            for name in all_squads
        ]
    )

    # ── One Tr per turn ──
    body_rows = []
    for turn in all_turns:
        turn_df = df[df.turn == turn]
        cells = [
            Td(
                Span(f"T{int(turn)}", 
                     style="font-size:12px;font-weight:800;font-family:monospace;"),
                style="padding:6px 8px;text-align:center;border-right:1px solid #eee;"
                      "vertical-align:middle;"
            )
        ]
        for name in all_squads:
            sq   = squads_by_name[name]
            rows = turn_df[turn_df.squad == name]
            if rows.empty:
                cells.append(Td(
                    Span("—", style="color:#ddd;"),
                    style=f"text-align:center;padding:8px;"
                          f"background:{sq.flag.lightComp}18;"
                          f"border-left:1px solid {sq.flag.lightComp};"
                ))
                continue

            # Multiple purchases in one turn stack vertically
            mini_cards = []
            for _, row in rows.iterrows():
                pt    = PieceType(row.piece_type)
                piece = sq._find_purchased_piece(log.parts, pt, int(row.hex_idx))
                if piece is not None:
                    mini_cards.append(_piece_mini(piece, log.parts))
                else:
                    mini_cards.append(DivVStacked(
                        NotStr(sq.flag.pieceLogo(pt, size=40)),
                        Span(row.piece_type.title(),
                             style="font-size:11px;font-weight:700;"),
                        Span(f"🍖{row.cost} · hex {row.hex_idx}",
                             style="font-size:10px;color:#888;"),
                        style="align-items:center;padding:4px;"
                    ))

            cells.append(Td(
                *mini_cards,
                style=f"padding:4px;vertical-align:top;"
                      f"background:{sq.flag.lightComp}18;"
                      f"border-left:2px solid {sq.flag.lightComp};"
            ))

        body_rows.append(Tr(
            *cells,
            style="border-bottom:1px solid #f5f5f5;"
        ))

    return Div(
        DivFullySpaced(
            H4("Purchases", cls="font-bold text-base"),
            Small(f"{len(df)} total · {len(all_turns)} turns",
                  cls=TextPresets.muted_sm),
            cls="mb-2"
        ),
        Div(
            Table(
                Thead(header),
                Tbody(*body_rows),
                style="border-collapse:collapse;width:100%;"
            ),
            style="overflow-x:auto;border:1px solid #e5e5e5;border-radius:12px;"
        ),
    )


In [ ]:
#| export
@patch
def purchaseTable(log: MoveLog) -> FT:
    """Turn × Squad grid. Each cell shows move_ft or blank."""
    if not log._moves:
        return Div(P("No moves recorded yet.", cls=TextPresets.muted_sm))

    squads = log.parts.squads
    all_turns = sorted({t for t, _, _ in log._moves})

    # Index: (turn, squad_name) → (squad, move)
    move_index = {}
    for t, sq, mv in log._moves:
        move_index[(t, sq.name)] = (sq, mv)

    # ── Header: one column per squad ──
    header = Tr(
        Th("Turn", style="padding:8px;font-size:11px;text-align:center;"),
        *[
            Th(
                DivVStacked(
                    NotStr(sq.flag.logo(size=28, piece_id=f"ph_{sq.name}")),
                    Span(sq.name,
                         style=f"font-size:10px;font-weight:700;"
                               f"color:{sq.flag.darkPrimary};"
                               f"text-align:center;max-width:100px;"),
                    style="align-items:center;gap:3px;"
                ),
                style=f"padding:8px;min-width:120px;text-align:center;"
                      f"background:{sq.flag.lightComp};"
                      f"border-bottom:2px solid {sq.flag.primary};"
            )
            for sq in squads
        ]
    )

    # ── One row per turn ──
    body_rows = []
    for turn in all_turns:
        cells = [
            Td(
                Span(f"T{int(turn)}",
                     style="font-size:12px;font-weight:800;font-family:monospace;"),
                style="padding:6px 8px;text-align:center;vertical-align:top;"
                      "border-right:1px solid #eee;"
            )
        ]

        for sq in squads:
            key = (turn, sq.name)
            if key not in move_index:
                # No move this turn for this squad
                cells.append(Td(
                    Div("—", style="color:#ddd;text-align:center;padding:24px 8px;"),
                    style=f"vertical-align:top;"
                          f"background:{sq.flag.lightComp}10;"
                          f"border-left:1px solid {sq.flag.lightComp};"
                ))
            else:
                squad, move = move_index[key]
                cells.append(Td(
                    log.parts.move_ft(move, squad=squad),
                    style=f"vertical-align:top;padding:4px;"
                          f"background:{sq.flag.lightComp}10;"
                          f"border-left:2px solid {sq.flag.lightComp};"
                ))

        body_rows.append(Tr(
            *cells,
            style="border-bottom:1px solid #f0f0f0;"
        ))

    return Div(
        DivFullySpaced(
            H4("Move Log", cls="font-bold text-base"),
            Small(f"{len(log._moves)} moves · {len(all_turns)} turns",
                  cls=TextPresets.muted_sm),
            cls="mb-2"
        ),
        Div(
            Table(
                Thead(header),
                Tbody(*body_rows),
                style="border-collapse:collapse;width:100%;"
            ),
            style="overflow-x:auto;border:1px solid #e5e5e5;border-radius:12px;"
        ),
    )


In [ ]:
def find_valid_placement_hex(parts: GameParts, squad: Squad) -> int | None:
    """Find an unoccupied land hex in any ally's sight."""
    kings = [p for p in squad.pieces if p.type == PieceType.KING and p.place is not None]
    if not kings: return None
    
    occupied = {
        parts.hp2i(p.place.location)
        for sq in parts.squads for p in sq.pieces
        if p.place is not None
    }
    
    placed = [p for p in squad.pieces if p.place is not None]
    visible = set()
    for ally in placed:
        bonus = ally.food.bonus_sight
        eff = max(2, ally.place.level) + ally.food.bonus_sight
        for hp in ally.place.location.field_of_view(ally.place.facing, eff):
            visible.add(parts.hp2i(hp))
    
    for idx in sorted(visible):
        if idx in occupied: continue
        if parts.terr.elevations[idx] <= 0: continue
        return idx
    return None

In [ ]:
def demo_moves(parts: GameParts, n_turns: int = 3) -> tuple['MoveLog', list]:
    """Simulate n_turns of placements across all squads, round-robin."""
    log = MoveLog(parts)
    events = []  # for display
    
    for turn in range(n_turns):
        for squad in parts.squads:
            # Find next unplaced non-king piece
            unplaced = [p for p in squad.pieces 
                        if p.place is None and p.type != PieceType.KING]
            if not unplaced: continue
            
            piece = unplaced[0]
            hex_idx = find_valid_placement_hex(parts, squad)
            if hex_idx is None: continue
            
            loc = parts.location_by_index(hex_idx)
            placement = Placement(piece, loc)
            
            try:
                move = Move(placement=placement)
                log.record(squad, move)
                events.append((turn, squad.name, piece.type, hex_idx))
            except MoveError as e:
                print(f"  skip {squad.name}: {e}")
    
    return log, events

In [ ]:
def demo_moves_with_purchases(parts: GameParts, n_turns: int = 4) -> MoveLog:
    """Simulate turns: place unplaced pieces + purchase when pantry allows."""
    log = MoveLog(parts)

    for turn in range(n_turns):
        for squad in parts.squads:
            placed  = [p for p in squad.pieces if p.place is not None]
            unplaced = [p for p in squad.pieces 
                        if p.place is None and p.type != PieceType.KING]

            placement = None
            purchase  = None
            rotations = []

            # ── Placement ──
            if unplaced:
                piece   = unplaced[0]
                hex_idx = find_valid_placement_hex(parts, squad)
                if hex_idx is not None:
                    placement = Placement(piece, parts.location_by_index(hex_idx))

            # ── Purchase: try cheapest affordable piece ──
            kings = [p for p in placed if p.type == PieceType.KING]
            if kings:
                pantry = kings[0].place.pantry
                affordable = [(pt, cost) for pt, cost in PURCHASE_COST.items()
                              if cost <= pantry]
                if affordable:
                    pt, _ = min(affordable, key=lambda x: x[1])
                    buy_idx = find_valid_placement_hex(parts, squad)
                    if buy_idx is not None:
                        buy_loc = parts.location_by_index(buy_idx)
                        purchase = Purchase(pt, buy_loc, buy_loc.facing)

            # ── Rotation: rotate first placed non-king ──
            dirs = HexPosition.directions()
            non_kings = [p for p in placed if p.type != PieceType.KING]
            if non_kings:
                p = non_kings[0]
                cur = dirs.index(p.place.facing) if p.place.facing in dirs else 0
                new_facing = dirs[(cur + 1) % 6]
                rotations = [Rotation(p, new_facing)]

            try:
                move = Move.from_squad(squad, rotations=rotations,
                                       placement=placement, purchase=purchase)
                log.record(squad, move, turn=turn)  # ← explicit turn
            except MoveError as e:
                try:
                    move = Move.from_squad(squad, placement=placement)
                    log.record(squad, move, turn=turn)  # ← same turn
                except MoveError:
                    pass

    return log


In [ ]:
def MoveDemo(parts: GameParts, n_turns: int = 4) -> FT:
    """Full demo: simulate moves then display log + purchase table."""
    log = demo_moves_with_purchases(parts, n_turns)

    # Move cards grid — one card per recorded move
    move_cards = [
        parts.move_ft(move, squad=squad)
        for _, squad, move in log._moves
    ]

    return Div(
        # ── Header ──
        DivFullySpaced(
            H3("Move Demo"),
            Small(f"{len(log._moves)} moves across {log.turn_count()} turns · "
                  f"{len(parts.squads)} squads",
                  cls=TextPresets.muted_sm),
            cls="mb-4"
        ),

        # ── Move cards ──
        H4("All Moves", cls="font-bold text-base mb-2"),
        Div(*move_cards,
            style="display:flex;flex-wrap:wrap;gap:8px;align-items:flex-start;"),

        Divider(),

        # ── Purchase table ──
        log.purchaseTable(),

        Divider(),

        # ── Full log ──
        log.moveLogDisplay(),

        cls="space-y-4 p-4"
    )


Should we refactor to use PendingMove

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — it would clean things up nicely. Right now the "move being built" state is scattered across:

1. **`MovePanel` params** — `selected_purchase_type`, `target_hex`, `pending_rotations`
2. **`demo_moves_with_purchases`** — local `rotations`, `placement`, `purchase` variables
3. **Route handlers** (not yet written) — would need to track the same state server-side

`PendingMove` consolidates all of that into one mutable object that lives in the session. Here's a cleaned-up version:

```python
@dataclass
class PendingMove:
    """Mutable accumulator for a move being built by the player."""
    squad: Squad | None = None
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None
    purchase: Purchase | None = None

    # ── UI-only transient state ──
    purchase_type: PieceType | None = None   # selected but not yet placed on map
    selected_piece: Piece | None = None      # piece picked for placement (awaiting hex click)

    def commit(self) -> Move:
        """Validate and produce a committed Move. Raises MoveError."""
        if self.squad is None:
            raise MoveError("no squad assigned")
        return Move.from_squad(self.squad,
            rotations=self.rotations,
            placement=self.placement,
            purchase=self.purchase)

    def clear(self):
        """Reset all pending state."""
        self.rotations.clear()
        self.placement = None
        self.purchase = None
        self.purchase_type = None
        self.selected_piece = None

    @property
    def has_actions(self) -> bool:
        return bool(self.rotations or self.placement or self.purchase)

    # ── Convenience builders ──
    def add_rotation(self, piece: Piece, facing: HexPosition):
        # Replace if already rotating this piece
        self.rotations = [r for r in self.rotations if r.piece != piece]
        self.rotations.append(Rotation(piece, facing))

    def remove_rotation(self, piece: Piece):
        self.rotations = [r for r in self.rotations if r.piece != piece]

    def set_placement(self, piece: Piece, location: 'MapPlace'):
        self.placement = Placement(piece, location)
        self.selected_piece = None

    def set_purchase(self, piece_type: PieceType, location: 'MapPlace', facing: HexPosition):
        self.purchase = Purchase(piece_type, location, facing)
        self.purchase_type = None
```

Then the downstream simplifications:

**`MovePanel`** becomes `MovePanel(pending: PendingMove, king_pantry: int)` — no more 5 scattered params.

**`MoveLog.record`** becomes:
```python
def record(self, pending: PendingMove, turn: int = None):
    move = pending.commit()          # validates
    move.apply(self.parts, pending.squad)
    # ... log to df + _moves ...
    pending.clear()                  # reset for next turn
```

**Route handlers** just mutate a single `PendingMove` stored in session state — each button click calls one method (`add_rotation`, `set_placement`, etc.) then returns the updated panel.

**Demo functions** also get cleaner:
```python
pending = PendingMove(squad=squad)
pending.set_placement(piece, loc)
pending.add_rotation(p, new_facing)
log.record(pending, turn=turn)
```

The main win is that `PendingMove` owns the "what has the player done so far this turn" question in one place, with a clean `commit()` boundary. Worth the refactor.

In [ ]:
myStuff = GameParts()
myStuff.loadSquads()

ourTroop = myStuff.squads[0]
theKing = ourTroop.pieces[-1]
show(theKing)

In [ ]:
show(MoveDemo(myStuff, n_turns=3))